In [18]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, concatenate, GaussianNoise, Dropout

# Simulation2

In [ ]:
# identity (35), position (100), allography (116)
X_id = pd.read_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_encodings.xlsx', index_col=0, sheet_name="identity").values.astype(np.float32)
X_pos = pd.read_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_encodings.xlsx', index_col=0, sheet_name="position").values.astype(np.float32)
X_allog = pd.read_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_encodings.xlsx', index_col=0,sheet_name="allog").values.astype(np.float32)
df_sem = pd.read_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/data/words_semantic_vec.xlsx', sheet_name='vectors')
Y_sem = df_sem.values.astype(np.float32)

in_id   = Input(shape=(35,),  name='in_id')
tower_id  = Dense(700, activation='sigmoid')(in_id)

in_pos  = Input(shape=(100,), name='in_pos')
#noisy_pos = GaussianNoise(stddev=0.2)(in_pos)                # inject positional noise
tower_pos = Dense(700, activation='sigmoid')(in_pos)

in_allog  = Input(shape=(116,), name='in_allog')
tower_allog = Dense(700, activation='sigmoid')(in_allog)


merged = concatenate([tower_id, tower_pos, tower_allog])
h = Dense(500, activation='sigmoid')(merged)

output = Dense(300, activation='sigmoid', name='sem_output')(h)

model2 = Model(inputs=[in_id, in_pos, in_allog], outputs=output)


model2.compile(optimizer='adam', loss='mse')
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)
model2.fit([X_id,X_pos,X_allog], Y_sem, batch_size=25, epochs=700, callbacks=[callback])
# model2.save('simulation_2_model.h5')

Epoch 1/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0104
Epoch 2/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - loss: 0.0028
Epoch 3/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - loss: 0.0028
Epoch 4/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0028
Epoch 5/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0028
Epoch 6/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - loss: 0.0028
Epoch 7/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - loss: 0.0028
Epoch 8/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - loss: 0.0028
Epoch 9/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - loss: 0.0028
Epoch 10/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - loss: 0.0028
Epoch 11/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - loss: 0.0028
Epoch 12/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0027
Epoch 13/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - loss: 0.0028
Epoch 14/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0028
Epoch 15/700
57

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
from pathlib import Path

data_lst = [
    'Large_Scale_PONG/new_filtered_fasttetx_vectors/data/TL_words_encodings.xlsx',
    'Large_Scale_PONG/new_filtered_fasttetx_vectors/data/DL_words_encodings.xlsx'
]

df_sim2 = pd.DataFrame()
for document in data_lst:
    key = Path(document).stem.replace('_words_encodings', '')

    X_id   = pd.read_excel(document, sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos  = pd.read_excel(document, sheet_name='position', index_col=0).values.astype(np.float32)
    X_allog = pd.read_excel(document, sheet_name='allog', index_col=0).values.astype(np.float32)
    pred = model2.predict([X_id, X_pos, X_allog])
    valz = []
    for i in range(len(Y_sem)):
      r = np.corrcoef(pred[i], Y_sem[i])[0,1]
      valz.append(round(1-r,3))


    df_sim2[f'target-{key}']= valz

df_sim2.to_excel('Large_Scale_PONG/new_filtered_fasttetx_vectors/results/PONG_simulation_2_correlation_8.xlsx',index=False)

451/451 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
